# Un script permettant de calculer des carte de likelihood pour des réseaux pré-entrainés sur Imagenet et testés sur le dataset Animal 10k


In [1]:
from retinotopy import *

Running on GPU :  NVIDIA GeForce RTX 3060 #GPU= 1
---------------------------------------------------------------------------------------------------
On date 2024-05-24, Running learning on host DESKTOP-27VNO0E with device cuda, pytorch==2.2.0+cu121
---------------------------------------------------------------------------------------------------
Welcome on Linux-5.10.16.3-microsoft-standard-WSL2-x86_64-with-glibc2.35
Random seed 1998 has been set.


In [2]:
# The dataset to import images from

data_set_type = 'animal_10k'
args = Params()
args.root = f'{DATAROOT}/{data_set_type}/ap-10k/' # Directory containing images
args.folders = ['animal_data'] # type of images to use

args.do_saccade = True
args.do_resize = False
args.do_mask = False

In [15]:
pos_pred_zero = ((args.resolution[0]*args.resolution[1])//2)

for model_data_set_type in data_set_types:
    print(50*'=')
    print(f'{model_data_set_type=}')    
    
    for model_name in  ['resnet18', 'resnet50', 'resnet101']:
        print(50*'.')
        
        for do_polar in [False, True]:

            args.do_polar = do_polar
            print(f'{args.do_polar=}')

            image_datasets = image_datasets_transforms(args, shuffle=False, verbose=False)
        
            print(50*'.')
            
            model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + '.pt'
            
            model = charge_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()

            annotations = get_annotation('animal')

            df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + f'_complete_heat_map_Animal_10k.parquet'
            print(df_filename)
            df_map = None
            
            with torch.no_grad():
                for i_image, (images, label) in tqdm(enumerate(image_datasets)):
                
                    image_name = image_datasets.imgs[i_image][0].split('/')[-1]
                    
                    ground_true_indices, three_points, ground_true = get_ground_true(args, image_name, annotations, 'Animal10k')
            
            
                    if ground_true.min() == 0.0:
                        since = time.time()
                        
                        images = images.to(device, non_blocking=True)
                        
                        heat_map = model(images)
        
        
                        _, pred_zero = torch.max(heat_map[pos_pred_zero].data, 0) # get the predicted 
                        
                        sum = 1 if torch.argmax(torch.sum(heat_map, dim=0)).item() in match['animal'] else 0
                        sum_in = 1 if torch.argmax(torch.sum(heat_map[ground_true_indices[0]], dim=0)).item() in match else 0
                        
                        heatmap_prior = torch.sum(torch.nn.functional.softmax(heat_map, dim=1)[:,match['animal']], dim=1)
    
                        mid_point, in_point, ext_point = get_like_point(heatmap_prior, args.resolution, three_points)
    
                        arg_max_prior = torch.argmax(heatmap_prior)
                        position_prior = (arg_max_prior.item()%args.resolution[0], arg_max_prior.item()//args.resolution[1])
                        pov_max_prior = heat_map[arg_max_prior]
                        _, pred_prior = torch.max(pov_max_prior.data, 0)
                        likelihood_prior = torch.max(heatmap_prior).item()
                    
    
                        arg_max_no_prior = torch.argmax(heatmap_soft)
                        pred_no_prior = (arg_max_no_prior%1000).item()
                        
                        
                        out_heat = th_delete(heatmap_prior, ground_true_indices[0])
                        
                        likelihood_out_max = torch.max(out_heat).item()
                        likelihood_in_max = torch.max(heatmap_prior[ground_true_indices[0]]).item()
    
                        likelihood_out_mean = torch.mean(out_heat).item()
                        likelihood_in_mean = torch.mean(heatmap_prior[ground_true_indices[0]]).item()
                        
                    
        
                        Iou = get_IoU(heatmap_prior.cpu(), ground_true.reshape(args.resolution[0]*args.resolution[1]))
                    
                        PG = 1 if likelihood_in_max > likelihood_out_max else 0
                        
    
                        df_map_ = pd.DataFrame({'ImageId':image_name, 'likelihood_in_max': likelihood_in_max, 'likelihood_out_max': likelihood_out_max,
                                                                     'pred_zero':pred_zero.item(), 'pred_prior':pred_prior.item(), 'pred_no_prior':pred_no_prior,
                                                                     'likelihood_prior':likelihood_prior, 'likelihood_zero':likelihood_zero, 'likelihood_no_prior':likelihood_no_prior,
                                                                     'likelihood_out_mean': likelihood_out_mean, 'likelihood_in_mean': likelihood_in_mean,
                                                                     'mid_point':mid_point.item(), 'in_point':in_point.item(), 'ext_point':ext_point.item(), 'Iou': [Iou], 'position_prior':[position_prior],
                                                                         'sum':sum,'sum_in':sum_in, 'time':time.time() - since, 'PG':PG})
                        df_map = store_pandas(df_map, df_map_)
            df_map.to_parquet(df_filename)
            


model_data_set_type='full'
..................................................
args.do_polar=False
..................................................
loading .... cached_data/2024-05-24_full_resnet18_cartesian.pt
cached_data/2024-05-24_full_resnet18_cartesian_complete_heat_map_Animal_10k.parquet


0it [00:00, ?it/s]


NameError: name 'ground_true_indices' is not defined